# Activity 3: Local Models with Hugging Face

**Week 6 Day 3 · Running model weights yourself, no API required**

**Run this notebook in Google Colab**, not locally. It downloads and runs real model weights, which wants a GPU that most laptops and the classroom VM do not have. Colab gives you one for free.

In Activity 1, every call left your machine: you sent text to OpenAI's or Google's servers and they sent text back. Ollama moved the *server* onto your own machine, but it was still a server you talked to over HTTP. This notebook removes the server entirely. You download a model's weights once, load them into memory yourself, and run the forward pass on your own hardware. No `base_url`, no API key, no per-token bill.

## What you will learn

- The difference between calling an API and loading weights yourself
- How to run a small, purpose-built model (sentiment classification) locally in seconds
- How to run a small local *chat* model and compare its output to the API calls from Activity 1
- Why data engineers care: cost at scale, offline processing, and data that legally cannot leave your infrastructure

---
## Colab setup

1. Open this notebook in Colab (**File -> Open notebook -> Upload**, or upload it to your Drive).
2. **Runtime -> Change runtime type -> T4 GPU**, then save. Everything below still works on CPU, just slower.
3. Colab is a disposable, pre-configured environment, not the repo-root UV project you use everywhere else. Use plain `pip` here, and only here, because that is what the Colab runtime expects. The `-U` matters: Colab ships its own `transformers`, and this notebook uses the current argument names, so you want the upgrade rather than whatever version the runtime happened to launch with.

In [ ]:
!pip install -q -U transformers accelerate

---
# 1. A purpose-built local model: sentiment classification

Not every local model is a chatbot. Most production uses of local models are small, specialized, and fast: one model that does exactly one job well. `pipeline(...)` from `transformers` is the shortest path from "model name" to "running inference."

In [ ]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert-base-uncased-finetuned-sst-2-english",
)

That line downloaded roughly 250MB of model weights once (cached for the rest of this Colab session) and loaded them into memory. Nothing was sent anywhere. Run it on a few insurance customer messages.

In [ ]:
messages = [
    "The adjuster called me back within an hour, really impressed with the service.",
    "Still waiting three weeks later for someone to even look at my claim.",
    "Repair authorization was approved, thanks for the quick turnaround.",
]

for m in messages:
    result = classifier(m)[0]
    print(f"{result['label']:>10} ({result['score']:.2f})  {m}")

You should see `POSITIVE` for the first and third messages and `NEGATIVE` for the second, each with a confidence score above 0.9. If a label looks wrong, keep the example, you will want it in a moment.

Each call ran entirely on this Colab machine's GPU (or CPU). There is no network round trip after the initial download, so this scales to thousands of messages a second with no per-call cost and no data leaving the machine, which matters a great deal once "message" becomes "claim note with a policyholder's name and address in it."

---
# 2. A local chat model

Sentiment classification is a narrow, single-purpose model. For something closer to what you did in Activity 1, chatting with a model in natural language, you need a general instruct-tuned model. `Qwen2.5-1.5B-Instruct` is small enough to run on Colab's free GPU and does not require any Hugging Face access request.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype=torch.float16,
    device_map="auto",
)
print("Model loaded on:", model.device)

This download is a few gigabytes, so the first run takes a minute or two. You should get back one sentence summarizing the collision claim. Expect it to be serviceable but blunter than the `gpt-4o-mini` version from Activity 1, and do not be surprised if it adds a little commentary the prompt never asked for. Once loaded, you build a prompt the same conceptual way as Activity 1: a list of role/content messages. The mechanics of turning that list into raw text the model understands (`apply_chat_template`) are handled for you.

In [ ]:
CLAIM_NOTE = (
    "Insured reports rear-end collision at low speed in a parking lot. "
    "Bumper cover cracked, no airbag deployment, other party's insurance "
    "already confirmed liability. Insured requests expedited repair "
    "authorization due to upcoming work travel."
)

chat = [
    {"role": "system", "content": "You are a claims operations assistant."},
    {"role": "user", "content": f"Summarize this claim note in one sentence for a supervisor:\n\n{CLAIM_NOTE}"},
]

prompt_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

output_ids = model.generate(**inputs, max_new_tokens=100)
response = tokenizer.decode(output_ids[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
print(response)

---
# 3. Same task, two very different tools

You just ran the identical claim note through `gpt-4o-mini` over the network (Activity 1) and through a 1.5-billion-parameter model on this Colab GPU. Put the two side by side and be honest about the gap.

| | OpenAI API (Activity 1) | Local `Qwen2.5-1.5B` (this notebook) |
|---|---|---|
| **Where it runs** | OpenAI's servers | This Colab GPU |
| **Cost per call** | Per token, billed | Free after the download |
| **Data leaves your machine?** | Yes | No |
| **Works offline?** | No | Yes, once weights are cached |
| **Model size** | Not disclosed, but far larger than this one | 1.5 billion parameters |
| **Answer quality** | Generally strong | Usable, but noticeably weaker on nuance |

Be precise about what that size row is and is not saying. `gpt-4o-mini` is itself a small, cheap model, not the largest thing OpenAI sells. So this is not a frontier-versus-tiny comparison, it is *cheap cloud model* versus *what fits on a free Colab GPU*, and the cheap cloud model still wins comfortably. The genuinely large models are bigger again, and none of them are offered as a download, which is the entire reason they are sold as a metered service.

The general rule holds: smaller models are measurably worse at reasoning, at following complex instructions, and at avoiding confident factual mistakes. Size is not the only thing that matters, a well-chosen small model beats a large one on a narrow task, but you cannot wish the gap away.

## Why a data engineer would choose local anyway

Quality is not the only axis that matters:

- **Data that cannot leave the building.** Some claim data, health information, or PII is contractually or legally restricted from being sent to a third-party API, no matter how good that API's privacy policy is. A local model is the only option.
- **Cost at scale.** Classifying ten million support tickets a month through a paid API adds up fast. A small local classifier (like the sentiment model in Part 1) that runs on hardware you already own can be dramatically cheaper per unit at high volume.
- **Latency and offline access.** No network round trip, and it keeps working if your internet does not.
- **Narrow, specialized tasks.** If the job is one thing done a million times (classify, extract a field, flag anomalies), a small fine-tuned model often matches a frontier model's accuracy on that one task for a fraction of the cost.

None of this makes local models a universal replacement for API calls. It makes them a second tool with a different cost and privacy profile, and the choice between them is a real engineering decision, not a preference.

---
# Your Turn

Work in your own copy of this notebook (in Colab, **File -> Save a copy in Drive**, or download and re-upload later).

1. Run the sentiment classifier from Part 1 on at least five messages of your own invention, a mix of clearly positive, clearly negative, and ambiguous ones. Note any it gets wrong.
2. Change the prompt sent to `Qwen2.5-1.5B-Instruct` in Part 2 to ask for the claim summary as exactly one clause, no more than 15 words. Does the small model follow that constraint as reliably as `gpt-4o-mini` did in Activity 1?

**Stretch goal:** time the local `model.generate(...)` call and compare it to the OpenAI API call's latency from Activity 1. Which is faster on Colab's free GPU, and would you expect that to change on a laptop CPU?

## What you did

- Ran a small, purpose-built local model (sentiment classification) with no API and no network call.
- Loaded and ran a general local chat model (`Qwen2.5-1.5B-Instruct`) from raw weights.
- Compared a local model's output quality directly against the frontier API call from Activity 1.
- Connected the trade-off (quality vs. cost, privacy, and offline access) to real data engineering decisions.

**Next:** [Activity 4](./Activity_4_Function_Calling_Internals.ipynb) goes back to the OpenAI client from Activity 1 and opens up what actually happens inside a model when it decides it needs to call a function.